# COVID-19 Data Cleaning

This notebook prepares the COVID-19 dataset for exploratory data analysis.

The original dataset is stored in `data/raw/` and is not modified.
The cleaned dataset will be exported to `data/processed/`.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(
    "../data/raw/covid_19_clean_complete.csv",
    parse_dates=["Date"]
)

df.head()

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.15330,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0,0,0,0,Africa


In [4]:
df.dtypes

Province/State               str
Country/Region               str
Lat                      float64
Long                     float64
Date              datetime64[us]
Confirmed                  int64
Deaths                     int64
Recovered                  int64
Active                     int64
WHO Region                   str
dtype: object

In [5]:
numeric_columns = [
    "Confirmed",
    "Deaths",
    "Recovered",
    "Active"
]

(df[numeric_columns] < 0).sum()

Confirmed     0
Deaths        0
Recovered     0
Active       18
dtype: int64

In [7]:
df["Calculated Active"] = (
    df["Confirmed"]
    - df["Deaths"]
    - df["Recovered"]
)

In [8]:
df["Active Difference"] = (
    df["Active"] - df["Calculated Active"]
)

In [9]:
df["Active Difference"].value_counts().head()

Active Difference
0    49068
Name: count, dtype: int64

In [10]:
df.drop(
    columns=["Calculated Active", "Active Difference"],
    inplace=True
)

### Identify all logically inconsistent records

#### Deaths + Recovered > Confirmed

In [11]:
inconsistent = df[
    df["Deaths"] + df["Recovered"] > df["Confirmed"]
]

inconsistent[
    [
        "Country/Region",
        "Date",
        "Confirmed",
        "Deaths",
        "Recovered",
        "Active"
    ]
]

,Country/Region,Date,Confirmed,Deaths,Recovered,Active
16238,China,2020-03-24,168,6,168,-6
16499,China,2020-03-25,168,6,168,-6
16760,China,2020-03-26,168,6,168,-6
17021,China,2020-03-27,168,6,168,-6
17282,China,2020-03-28,168,6,168,-6
17543,China,2020-03-29,168,6,168,-6
17804,China,2020-03-30,168,6,168,-6
18065,China,2020-03-31,168,6,168,-6
18326,China,2020-04-01,168,6,168,-6
32059,United Kingdom,2020-05-23,558,45,515,-2


In [12]:
len(inconsistent)

18

In [13]:
df["Data Inconsistent"] = (
    df["Deaths"] + df["Recovered"] > df["Confirmed"]
)

We don't have the original source records needed to determine the correct value.

So the scientifically defensible approach is:

### Keep the original observations, but flag them.

Create:

In [14]:
df["Data Inconsistent"] = (
    df["Deaths"] + df["Recovered"] > df["Confirmed"]
)

In [15]:
df["Data Inconsistent"].value_counts()

Data Inconsistent
False    49050
True        18
Name: count, dtype: int64

### Check the other numerical problems

In [16]:
numeric_columns = [
    "Confirmed",
    "Deaths",
    "Recovered",
    "Active"
]

print("Negative values:")
print((df[numeric_columns] < 0).sum())

print("\nDeaths > Confirmed:")
print((df["Deaths"] > df["Confirmed"]).sum())

print("\nDeaths + Recovered > Confirmed:")
print(
    (df["Deaths"] + df["Recovered"] > df["Confirmed"]).sum()
)

Negative values:
Confirmed     0
Deaths        0
Recovered     0
Active       18
dtype: int64

Deaths > Confirmed:
0

Deaths + Recovered > Confirmed:
18


### verify that our flag catches exactly the negative Active cases:

In [19]:
(
    df['Data Inconsistent'] == (df['Active'] < 0)
).all()

np.True_

### Check for duplicate country/date combinations

In [20]:
country_date_counts = (
    df.groupby(["Country/Region", "Date"])
      .size()
)

country_date_counts.value_counts().sort_index()

1     33840
2       188
4       188
8       188
11      376
12      188
33      188
Name: count, dtype: int64

In [21]:
(
    df.groupby("Country/Region")["Province/State"]
      .apply(lambda x: x.notna().sum())
      .sort_values(ascending=False)
      .head(20)
)

Country/Region
China                  6204
Canada                 2256
France                 1880
United Kingdom         1880
Australia              1504
Netherlands             564
Denmark                 188
Greenland               188
Algeria                   0
Antigua and Barbuda       0
Angola                    0
Bahamas                   0
Bahrain                   0
Bangladesh                0
Barbados                  0
Belarus                   0
Belgium                   0
Belize                    0
Benin                     0
Bhutan                    0
Name: Province/State, dtype: int64

In [23]:
df.to_csv(
    "../data/processed/covid_cleaned.csv",
    index=False
)